In [1]:
import sys
from pathlib import Path

# Raíz del proyecto (ejecutar desde notebooks/)
sys.path.append("..")

import json
import pandas as pd
from datetime import datetime

from src.core.price_tracking import run_price_tracking
pd.set_option('display.max_colwidth', None)

In [2]:
# Cargar URLs por marca desde config (ruta relativa al proyecto)
CONFIG_PATH = Path.cwd().parent / "src" / "data" / "json" / "price_tracking_urls.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path.cwd() / "src" / "data" / "json" / "price_tracking_urls.json"

with open(CONFIG_PATH, encoding="utf-8") as f:
    url_config = json.load(f)

# Ejemplo: url_config = {"italika": ["https://...", "https://..."]}
print("Marcas en config:", list(url_config.keys()))
for brand, urls in url_config.items():
    print(f"  {brand}: {len(urls)} URLs")

Marcas en config: ['italika']
  italika: 45 URLs


In [3]:
# Modo: "json" (5 cr/pág, schema) o "markdown" (sin costo extra, parseo regex)
# Ejecutar ambos modos y concatenar en un solo DataFrame
EXTRACTION_MODES = ["markdown"] # "json", "markdown"
all_rows = []

for brand_name, urls in url_config.items():
    for mode in EXTRACTION_MODES:
        rows = run_price_tracking(
            urls=urls,
            brand_name=brand_name,
            mode=mode,
            tag=f"{brand_name}-prices",
        )
        all_rows.extend(rows)

df = pd.DataFrame(all_rows)
# Orden de columnas (incl. precio base, neto y descuento para modo JSON)
COLUMNS = [
    "brand_name", "model_name", "url", "change_status", "previous_scrape_at",
    "visibility",
    "price_base_current", "price_net_current", "price_discount_current",
    "price_base_previous", "price_net_previous", "price_discount_previous",
    "price_current", "price_previous",
    "availability_current", "availability_previous", "extraction_mode",
]
df = df[[c for c in COLUMNS if c in df.columns]]
df

,brand_name,model_name,url,change_status,previous_scrape_at,visibility,price_base_current,price_net_current,price_discount_current,price_base_previous,price_net_previous,price_discount_previous,price_current,price_previous,availability_current,availability_previous,extraction_mode
0,italika,motoneta italika d150 lt negra,https://www.italika.mx/motoneta-italika-d150-lt-negra-34005388/p,same,2026-03-17T17:12:16.689+00:00,visible,,"18,499",,,,,"18,499",,Disponible,,markdown
1,italika,motoneta italika ws150 sport marino,https://www.italika.mx/motoneta-italika-ws150-sport-marino-34006551/p,same,2026-03-17T17:12:16.426+00:00,visible,,"23,999",,,,,"23,999",,Disponible,,markdown
2,italika,motoneta italika ws175 sport grafito,https://www.italika.mx/motoneta-italika-ws175-sport-grafito-34006561/p,same,2026-03-17T17:12:16.564+00:00,visible,,"25,999",,,,,"25,999",,Disponible,,markdown
3,italika,motoneta italika vitalia 150 beige con gps,https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p,same,2026-03-17T17:12:15.801+00:00,visible,,"29,999",,,,,"29,999",,Disponible,,markdown
4,italika,motoneta italika d150 blanco con azul,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,same,2026-03-17T17:12:17.212+00:00,visible,,"18,999",,,,,"18,999",,Disponible,,markdown
5,italika,motoneta italika modena 175 con gps azul,https://www.italika.mx/motoneta-italika-modena-175-con-gps-azul-34006716/p,same,2026-03-17T17:12:25.33+00:00,visible,,"29,999",,,,,"29,999",,Disponible,,markdown
6,italika,motoneta italika bit 150 blanca con negro,https://www.italika.mx/motoneta-italika-bit-150-blanca-con-negro-34006931/p,same,2026-03-17T18:00:37.267+00:00,visible,,"26,999",,,,,"26,999",,Disponible,,markdown
7,italika,motoneta italika d125 azul con negro,https://www.italika.mx/motoneta-italika-d125-azul-con-negro-34006289/p,same,2026-03-17T18:00:37.347+00:00,visible,,"16,499",,,,,"16,499",,Disponible,,markdown
8,italika,motoneta italika ds150 negra,https://www.italika.mx/motoneta-italika-ds150-negra-34006307/p,same,2026-03-17T18:00:39.786+00:00,visible,,"22,999",,,,,"22,999",,Disponible,,markdown
9,italika,cuatrimoto italika atv300 azul,https://www.italika.mx/cuatrimoto-italika-atv300-azul-34006717/p,same,2026-03-17T18:00:39.519+00:00,visible,,"79,999",,,,,"79,999",,Disponible,,markdown


In [20]:
import re

def clean_model_name(model):
    # 1. Eliminar todo hasta e incluyendo 'Italika'
    match = re.search(r"Italika\s*(.*)", model, re.IGNORECASE)
    if match:
        model = match.group(1).strip()
    # 2. Eliminar palabras como "con" (usualmente usada como con GPS, etc)
    model = re.sub(r'\bcon\b', '', model, flags=re.IGNORECASE)
    # 3. Eliminar colores (trituramos lista de colores más comunes, todas las variantes de blanco, negro, rojo, etc)
    colores = [
        'blanca', 'blanco', 'negro', 'negra', 'azul', 'rojo', 'roja', 'verde', 'gris',
        'amarillo', 'amarilla', 'naranja', 'dorado', 'dorada', 'plateado', 'plateada',
        'cafe', 'café', 'morado', 'morada', 'rosado', 'rosada', 'beige', 'vino', 'plata',
        'perla', 'caramelo', 'grafito', 'anaranjado', 'fucsia', 'lila', 'mate', 'carbono',
        'camuflaje', 'camuflado', 'turquesa'
    ]
    # Quitar color si es la última palabra, o lista separada con espacio o coma
    pattern_colores = r'(?:\b(?:' + '|'.join(colores) + r')\b\.?,?\s*)'
    # Eliminar colores al final y también antes del final (puede haber más de uno)
    model = re.sub(pattern_colores + r'*$', '', model, flags=re.IGNORECASE)
    model = re.sub(pattern_colores, '', model, flags=re.IGNORECASE)
    # Normalizar espacios múltiples
    model = re.sub(r'\s+', ' ', model)
    return model.strip()

# Aplicar limpieza a la columna model_name y guardar resultado en una nueva columna
df['model_name_clean'] = df['model_name'].apply(clean_model_name)

In [ ]:
df = df[['brand_name',
 'model_name',
 'model_name_clean',
 'url',
 'change_status',
 'previous_scrape_at',
 'visibility',
 'price_base_current',
 'price_net_current',
 'price_discount_current',
 'price_base_previous',
 'price_net_previous',
 'price_discount_previous',
 'price_current',
 'price_previous',
 'availability_current',
 'availability_previous',
 'extraction_mode',
 ]]

In [ ]:
# # Guardar CSV (UTF-8, BOM opcional para Excel)
# OUTPUT_DIR = Path.cwd().parent / "output"
# OUTPUT_DIR.mkdir(exist_ok=True)
# filename = f"price_tracking_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
# out_path = OUTPUT_DIR / filename
# df.to_csv(out_path, index=False, encoding="utf-8-sig", sep=",")
# print(f"Guardado: {out_path}")

Guardado: c:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\scrape_websites_refactorv2\output\price_tracking_20260317_1412.csv


In [11]:
df_mapped = pd.read_csv(r"C:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\scrape_websites_refactorv2\output\price_tracking_20260317_1412.csv")

In [4]:
df_inventory = pd.read_csv(r"C:\Users\JTRUJILLO\Documents\Galgo\Scripts\Data\historical_data\src\data\prices\MX-prices.csv")

In [12]:
df_scraped_brand_filtered = df_mapped[df_mapped["brand_name"] == "italika"]

In [13]:
df_inventory_brand = df_inventory[df_inventory["brand"] == "Italika"]

In [16]:
# Recalcular merge con inventario de forma segura (sobrescribe df_result anterior)
# Normalizamos a string antes de pasar a minúsculas para evitar problemas con NaN/u otros tipos

df_mapped["model_lower"] = df_mapped["model_name"].astype(str).str.lower()

_df_inventory_copy = df_inventory.copy()
_df_inventory_copy["model_lower"] = _df_inventory_copy["model"].astype(str).str.lower()

inventory_cols = ["code", "model_lower", "model", "year"]
_df_inv_subset = (
    _df_inventory_copy[inventory_cols]
    .drop_duplicates()
    .rename(columns={"model": "model_inventory"})
)

df_result = pd.merge(
    df_mapped,
    _df_inv_subset,
    on=["model_lower", "year"],
    how="left",
    indicator=True,
)

# Usar nombre del inventario cuando hubo match (capitalización correcta del sistema)
df_result["model"] = df_result["model_inventory"].fillna(df_result["model"])
df_result.drop(columns=["model_lower", "model_inventory"], inplace=True)

KeyError: 'year'

In [7]:
df_scraped_brand_filtered.columns

Index(['brand_name', 'model_name', 'model_name_clean', 'url', 'change_status',
       'previous_scrape_at', 'visibility', 'price_base_current',
       'price_net_current', 'price_discount_current', 'price_base_previous',
       'price_net_previous', 'price_discount_previous', 'price_current',
       'price_previous', 'availability_current', 'availability_previous',
       'extraction_mode'],
      dtype='str')

In [8]:
from src.utils.replace_model_name import map_model_name, load_mapping_file, save_mapping_file, normalize_brand_name
from src.config.settings import COUNTRY
mapeo = load_mapping_file(COUNTRY, "Italika")

df_mapped = df_scraped_brand_filtered.copy()
df_mapped["model_original"] = df_mapped["model_name"]
df_mapped["model"] = df_mapped["model_name"].apply(lambda x: map_model_name(x, mapeo))

# Verificar qué modelos fueron mapeados
df_cambios = df_mapped[df_mapped["model"] != df_mapped["model_original"]]
if len(df_cambios) > 0:
    print(f"📝 {len(df_cambios)} modelo(s) fueron renombrados por el mapeo JSON:")
    print(df_cambios[["model_original", "model"]].drop_duplicates().to_string(index=False))
else:
    print("ℹ️ Ningún modelo fue renombrado (todos coinciden con inventario o no están en el JSON)")

Archivo de mapeo cargado correctamente: ../src/data/json/replace_name/MX/italika_mapeo_nombres.json
ℹ️ Ningún modelo fue renombrado (todos coinciden con inventario o no están en el JSON)


In [10]:
# ============================================================
# PASO 2: Un solo merge con inventario (lowercase para evitar problemas de capitalización)
# ============================================================
df_mapped["model_lower"] = df_mapped["model"].str.lower()

df_inventory_copy = df_inventory.copy()
df_inventory_copy["model_lower"] = df_inventory_copy["model"].str.lower()

# Columnas que necesitamos del inventario
inventory_cols = ["code", "model_lower", "model", "year"]
df_inv_subset = df_inventory_copy[inventory_cols].drop_duplicates().rename(columns={"model": "model_inventory"})

df_result = pd.merge(
    df_mapped,
    df_inv_subset,
    on=["model_lower", "year"],
    how="left",
    indicator=True
)

# Usar nombre del inventario cuando hubo match (capitalización correcta del sistema)
df_result["model"] = df_result["model_inventory"].fillna(df_result["model"])
df_result.drop(columns=["model_lower", "model_inventory"], inplace=True)

KeyError: 'year'